# Himawari L2 valid-retrieval availability over Vietnam

For every Stage-A gridded slot that falls inside Stage B's **daytime observation window**, count the fraction of Vietnam-land pixels where `AOD_himawari_l2` is a valid (non-NaN) retrieval.

**Definition.** Per pixel: *valid* ⇔ `~isnan(AOD_himawari_l2)` (Stage A already applied JAXA bit-mask + `HIMAWARI_RF_MIN`/`HIMAWARI_UNC_MAX`/strict-zero gate).

**Denominator.** Slots are taken from `stage_b/slots.iter_window_slots(day)` — the same window Stage B's `coverage_audit` uses. The window is data-driven: for each UTC day it is `[min(HHMM), max(HHMM)]` over Stage A merged files on disk, with a 7-day median fallback when <10 slots are present (see `stage_b/slots.discover_day_window`).

**Pooling.** Per month-of-year $m$ and region $r$:
$$\text{avail}_{m,r}=\frac{\sum_t \#\{(i,j)\in\text{mask}_r:\,\text{valid}_t(i,j)\}}{|\text{mask}_r|\cdot N_t(m)}$$
where the slot sum runs over every in-window slot whose UTC month is $m$, across all available years.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
from datetime import date, datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from tqdm.auto import tqdm

# Stage B's config re-exports the Stage A grid + adds the window-discovery
# constants used by stage_b.slots.  Importing stage_b first means `from config
# import ...` inside slots.py resolves to the right module.
_STAGE_B = Path('../stage_b').resolve()
if str(_STAGE_B) not in sys.path:
    sys.path.insert(0, str(_STAGE_B))

from config import LATS, LONS, NLAT, NLON, NORTH_CENTRAL_LAT, CENTRAL_SOUTH_LAT
from slots import iter_local_days, iter_window_slots

GRIDDED_DIR = Path('/home/slow_data/Air_Quality/Stage_A/gridded')
GADM_VN_L0  = Path('/home/work1/projects/Air_Quality/GADM_Vietnam/gadm41_VNM_0.shp')

# Sweep the full data span.  Days outside the actual coverage yield no slots
# (discover_day_window returns None) so the bounds can be safely loose.
DATE_START = date(2022, 9, 1)
DATE_END   = date(2026, 4, 30)

print('Grid:', NLAT, 'x', NLON, ' Gridded dir:', GRIDDED_DIR)

In [ ]:
# warnings.filterwarnings('ignore')
# sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
# plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})


## 1. Vietnam land mask on the production grid

Rasterize GADM Vietnam (level-0 country polygon) onto the same 310×160 grid Stage A uses, by point-in-polygon test on cell centres.

In [ ]:
import geopandas as gpd
from shapely.vectorized import contains

gdf_vn = gpd.read_file(GADM_VN_L0)
vn_geom = gdf_vn.geometry.union_all() if hasattr(gdf_vn.geometry, 'union_all') else gdf_vn.geometry.unary_union

lon2d, lat2d = np.meshgrid(LONS, LATS)
vn_mask = contains(vn_geom, lon2d, lat2d)

print(f'Vietnam-land pixels: {vn_mask.sum()} of {vn_mask.size} '
      f'({100*vn_mask.mean():.2f}% of bbox)')

In [ ]:
region_masks = {
    'north':   vn_mask & (lat2d >= NORTH_CENTRAL_LAT),
    'central': vn_mask & (lat2d <  NORTH_CENTRAL_LAT) & (lat2d >= CENTRAL_SOUTH_LAT),
    'south':   vn_mask & (lat2d <  CENTRAL_SOUTH_LAT),
    'vietnam': vn_mask,
}
for r, m in region_masks.items():
    print(f'  {r:8s}: {m.sum():5d} pixels')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6), sharey=True)
for ax, (name, m) in zip(axes, region_masks.items()):
    ax.imshow(m, extent=[LONS.min(), LONS.max(), LATS.min(), LATS.max()],
              origin='upper', cmap='Greens')
    ax.set_title(f'{name}  (n={m.sum()})')
    ax.set_xlabel('lon')
axes[0].set_ylabel('lat')
plt.tight_layout(); plt.show()

## 2. Aggregate over Stage B's daytime windows

For each local day in `[DATE_START, DATE_END]`, walk the slots returned by `iter_window_slots(day)`.  Each slot maps to a gridded file at `GRIDDED_DIR/YYYY/MM/DD/gridded_YYYYMMDD_HHMM.nc`.

In [ ]:
import netCDF4 as nc

def gridded_path(slot_utc: datetime) -> Path:
    return (GRIDDED_DIR
            / f'{slot_utc.year:04d}'
            / f'{slot_utc.month:02d}'
            / f'{slot_utc.day:02d}'
            / f'gridded_{slot_utc:%Y%m%d_%H%M}.nc')

region_pix = {r: int(m.sum()) for r, m in region_masks.items()}
valid_sum  = {r: np.zeros(13, dtype=np.int64) for r in region_masks}  # index 1..12
slots_used      = np.zeros(13, dtype=np.int64)
skipped_no_file = 0
skipped_no_var  = 0
skipped_error   = 0
first_error     = None

days = list(iter_local_days(DATE_START, DATE_END))
for d in tqdm(days, mininterval=2.0):
    for slot_utc in iter_window_slots(d):
        p = gridded_path(slot_utc)
        if not p.exists():
            skipped_no_file += 1
            continue
        try:
            with nc.Dataset(p) as ds:
                if 'AOD_himawari_l2' not in ds.variables:
                    skipped_no_var += 1
                    continue
                arr = np.ma.filled(
                    ds.variables['AOD_himawari_l2'][:].astype(np.float32),
                    np.nan,
                )
        except Exception as e:
            skipped_error += 1
            if first_error is None:
                first_error = (str(p), repr(e))
            continue

        valid = ~np.isnan(arr)
        month = slot_utc.month
        slots_used[month] += 1
        for r, m in region_masks.items():
            valid_sum[r][month] += int((valid & m).sum())

print(f'\nslots used (sum across months):    {slots_used.sum():,}'
      f'\nslots in window but file missing:  {skipped_no_file:,}'
      f'\nfiles missing AOD_himawari_l2 var: {skipped_no_var:,}'
      f'\nfiles that raised on open:         {skipped_error:,}')
if first_error is not None:
    print(f'\nfirst open error:\n  path = {first_error[0]}\n  err  = {first_error[1]}')

## 3. Build the availability table

In [ ]:
MONTHS = ['January','February','March','April','May','June',
         'July','August','September','October','November','December']

rows = []
for mi, name in enumerate(MONTHS, start=1):
    n_slots = int(slots_used[mi])
    row = {'Month': name, 'slots': n_slots}
    for r in ('vietnam', 'north', 'central', 'south'):
        denom = region_pix[r] * n_slots
        row[r] = 100.0 * valid_sum[r][mi] / denom if denom else np.nan
    rows.append(row)

annual = {'Month': 'Annual', 'slots': int(slots_used[1:].sum())}
for r in ('vietnam', 'north', 'central', 'south'):
    denom = region_pix[r] * annual['slots']
    annual[r] = 100.0 * valid_sum[r][1:].sum() / denom if denom else np.nan
rows.append(annual)

df = pd.DataFrame(rows).set_index('Month')
df = df[['slots', 'vietnam', 'north', 'central', 'south']]
df.columns = ['slots', 'Vietnam pooled (%)', 'North (%)', 'Central (%)', 'South (%)']
df.round(2)

In [ ]:
# def fmt(v):
#     return '--' if pd.isna(v) else f'{v:.2f}'

# print('\\textbf{Month} & \\textbf{Vietnam pooled (\\%)}'
#       ' & \\textbf{North (\\%)} & \\textbf{Central (\\%)} & \\textbf{South (\\%)} \\\\')
# print('\\hline')
# for name in MONTHS:
#     r = df.loc[name]
#     print(f"{name:<9} & {fmt(r['Vietnam pooled (%)'])} & {fmt(r['North (%)'])} & "
#           f"{fmt(r['Central (%)'])} & {fmt(r['South (%)'])} \\\\")
# print('\\hline')
# r = df.loc['Annual']
# print(f"\\textbf{{Annual}} & {fmt(r['Vietnam pooled (%)'])} & {fmt(r['North (%)'])} & "
#       f"{fmt(r['Central (%)'])} & {fmt(r['South (%)'])} \\\\")
# print('\\hline')

## 4. Raw per-year slot availability

A coarser, no-mask variant: **a slot counts as 1 if `AOD_himawari_l2` has any finite pixel anywhere on the grid**, divided by the platonic total of slots in that calendar year (`48 × days_in_year` — no daytime-window discovery, no Vietnam mask).

In [ ]:
# import calendar
# from datetime import timedelta

# YEARS = list(range(2022, 2027))

# per_year_rows = []
# for yr in YEARS:
#     days_in_year = 366 if calendar.isleap(yr) else 365
#     total_slots = 48 * days_in_year
#     n_valid = 0
#     n_files = 0

#     d = date(yr, 1, 1)
#     end = date(yr, 12, 31)
#     pbar = tqdm(total=days_in_year, desc=f'{yr}', mininterval=2.0)
#     while d <= end:
#         for slot_idx in range(48):
#             h, half = divmod(slot_idx, 2)
#             slot_utc = datetime(d.year, d.month, d.day, h, half * 30)
#             p = gridded_path(slot_utc)
#             if not p.exists():
#                 continue
#             try:
#                 with nc.Dataset(p) as ds:
#                     if 'AOD_himawari_l2' not in ds.variables:
#                         continue
#                     arr = np.ma.filled(
#                         ds.variables['AOD_himawari_l2'][:].astype(np.float32),
#                         np.nan,
#                     )
#             except Exception:
#                 continue
#             n_files += 1
#             if np.isfinite(arr).any():
#                 n_valid += 1
#         d += timedelta(days=1)
#         pbar.update(1)
#     pbar.close()

#     per_year_rows.append({
#         'year':              yr,
#         'days_in_year':      days_in_year,
#         'total_slots':       total_slots,
#         'files_on_disk':     n_files,
#         'slots_with_valid':  n_valid,
#         'availability_%':    100.0 * n_valid / total_slots,
#     })

# df_year = pd.DataFrame(per_year_rows).set_index('year')
# df_year.round(2)

## 5. Month × slot-of-day heatmap (Himawari L2 over Vietnam)

For Sep 2022 – Apr 2026 (Stage A's full operational span), build a 12-row × 48-column heatmap of the **mean valid-retrieval fraction over Vietnam**: for each (month-of-year, UTC slot-of-day), average across all observed slots of `#valid_VN_pixels / #VN_pixels`. Trim columns to those with any observation (~22 columns of daytime data). Contours at 5%, 10%, 20%.

In [ ]:
HEAT_START = date(2022, 9, 1)
HEAT_END   = date(2026, 4, 30)

vn_n = int(vn_mask.sum())
heat_frac_sum = np.zeros((13, 48), dtype=np.float64)  # sum of per-slot valid fractions
heat_count    = np.zeros((13, 48), dtype=np.int64)    # n slots observed

for d in tqdm(list(iter_local_days(HEAT_START, HEAT_END)), mininterval=2.0):
    for slot_idx in range(48):
        h, half = divmod(slot_idx, 2)
        slot_utc = datetime(d.year, d.month, d.day, h, half * 30)
        p = gridded_path(slot_utc)
        if not p.exists():
            continue
        try:
            with nc.Dataset(p) as ds:
                if 'AOD_himawari_l2' not in ds.variables:
                    continue
                arr = np.ma.filled(
                    ds.variables['AOD_himawari_l2'][:].astype(np.float32),
                    np.nan,
                )
        except Exception:
            continue
        valid_frac = float((np.isfinite(arr) & vn_mask).sum()) / vn_n
        heat_frac_sum[d.month, slot_idx] += valid_frac
        heat_count[d.month, slot_idx]    += 1

# mean fraction per (month, slot), in %
with np.errstate(invalid='ignore', divide='ignore'):
    heat_mean = 100.0 * heat_frac_sum[1:, :] / heat_count[1:, :]   # (12, 48)

col_mask = heat_count[1:, :].sum(axis=0) > 0
cols = np.where(col_mask)[0]
H = heat_mean[:, cols]   # (12, n_cols)

print(f'columns with any data: {len(cols)}  (slot-idx range {cols[0]}..{cols[-1]})')
print(f'value range: {np.nanmin(H):.2f}% .. {np.nanmax(H):.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

# Clamp colour scale to a meaningful upper bound — the wet-season trough is the
# story, not the high-altitude maximum.
vmax = max(20.0, float(np.nanpercentile(H, 95)))
im = ax.imshow(H, aspect='auto', cmap='viridis', origin='upper', vmin=0, vmax=vmax)

xt_labels = [f'{c//2:02d}:{(c%2)*30:02d}' for c in cols]
ax.set_xticks(np.arange(len(cols)))
ax.set_xticklabels(xt_labels, rotation=60, ha='right')
ax.set_yticks(np.arange(12))
ax.set_yticklabels(MONTHS)
ax.set_xlabel('UTC slot-of-day  (Vietnam local = UTC+7)')
ax.set_ylabel('Month-of-year')
ax.set_title(f'Himawari L2 valid-retrieval fraction over Vietnam '
             f'({HEAT_START:%Y-%m} → {HEAT_END:%Y-%m})')

# Contour overlays at 5%, 10%, 20%
xx, yy = np.meshgrid(np.arange(len(cols)), np.arange(12))
H_for_contour = np.where(np.isfinite(H), H, 0.0)
cs = ax.contour(xx, yy, H_for_contour, levels=[5, 10, 20],
                colors='white', linewidths=1.1)
ax.clabel(cs, fmt='%d%%', fontsize=9)

cb = plt.colorbar(im, ax=ax, pad=0.02)
cb.set_label('Mean valid-retrieval fraction (%)')

plt.tight_layout()
plt.show()

## 6. Per-LEO availability companion table

Same shape as §3, but for MODIS MAIAC, VIIRS SNPP, and VIIRS NOAA-20 — reported as **fraction of satellite-overpass days carrying a finite retrieval**. Period: Sep 2022 – Apr 2026.

For each sensor `S` and local UTC day `d`:
- *overpass day* ⇔ at least one 30-min slot in `d` has variable `AOD_S` present in its Stage-A **merged** file (the variable is only written when the swath crossed Vietnam).
- *finite at pixel `(i,j)`* ⇔ on `d`, at least one slot with `AOD_S` reported a finite value at `(i,j)`.

Per month-of-year × region:
$$\text{avail}_{m,r,S}=\frac{\sum_{d\in\text{overpass}_S(m)}\#\{(i,j)\in\text{mask}_r:\text{finite}_{S,d}(i,j)\}}{|\text{mask}_r|\cdot |\text{overpass}_S(m)|}$$

This is pooled-region accounting (denominator = region pixels × overpass-days). Per-pixel `n_overpass` can't be cleanly recovered from merged files — NaN there could mean swath-missed or swath-hit-but-failed — and the pooled form already answers the LEO-rescue question without that confound.

In [ ]:
MERGED_DIR = Path('/home/slow_data/Air_Quality/Stage_A/gridded')
LEOS = ['modis_maiac', 'viirs_snpp', 'viirs_noaa20']

LEO_START = HEAT_START  # date(2022, 9, 1)
LEO_END   = HEAT_END    # date(2026, 4, 30)

def merged_path(slot_utc: datetime) -> Path:
    return (MERGED_DIR
            / f'{slot_utc.year:04d}'
            / f'{slot_utc.month:02d}'
            / f'{slot_utc.day:02d}'
            / f'gridded_{slot_utc:%Y%m%d_%H%M}.nc')

n_overpass = {s: np.zeros(13, dtype=np.int64) for s in LEOS}
n_finite   = {s: {r: np.zeros(13, dtype=np.int64) for r in region_masks} for s in LEOS}

for d in tqdm(list(iter_local_days(LEO_START, LEO_END)), mininterval=2.0):
    daily_any_finite = {s: np.zeros((NLAT, NLON), dtype=bool) for s in LEOS}
    daily_overpass   = {s: False for s in LEOS}
    for slot_idx in range(48):
        h, half = divmod(slot_idx, 2)
        slot_utc = datetime(d.year, d.month, d.day, h, half * 30)
        p = merged_path(slot_utc)
        if not p.exists():
            continue
        try:
            with nc.Dataset(p) as ds:
                for s in LEOS:
                    var = f'AOD_{s}'
                    if var in ds.variables:
                        daily_overpass[s] = True
                        arr = np.ma.filled(
                            ds.variables[var][:].astype(np.float32),
                            np.nan,
                        )
                        daily_any_finite[s] |= np.isfinite(arr)
        except Exception:
            continue

    m = d.month
    for s in LEOS:
        if not daily_overpass[s]:
            continue
        n_overpass[s][m] += 1
        for r, mask in region_masks.items():
            n_finite[s][r][m] += int((daily_any_finite[s] & mask).sum())

print('overpass days per sensor (annual):',
      {s: int(n_overpass[s][1:].sum()) for s in LEOS})

In [ ]:
def _leo_table(s: str) -> pd.DataFrame:
    rows = []
    for mi, name in enumerate(MONTHS, start=1):
        nd = int(n_overpass[s][mi])
        row = {'Month': name, 'overpass_days': nd}
        for r in ('vietnam', 'north', 'central', 'south'):
            denom = region_pix[r] * nd
            row[r] = 100.0 * n_finite[s][r][mi] / denom if denom else np.nan
        rows.append(row)
    nd_t = int(n_overpass[s][1:].sum())
    annual = {'Month': 'Annual', 'overpass_days': nd_t}
    for r in ('vietnam', 'north', 'central', 'south'):
        denom = region_pix[r] * nd_t
        annual[r] = 100.0 * sum(n_finite[s][r][1:]) / denom if denom else np.nan
    rows.append(annual)
    df_s = pd.DataFrame(rows).set_index('Month')
    df_s = df_s[['overpass_days', 'vietnam', 'north', 'central', 'south']]
    df_s.columns = ['overpass days', 'Vietnam (%)', 'North (%)',
                    'Central (%)', 'South (%)']
    return df_s

sensor_dfs = {s: _leo_table(s) for s in LEOS}
for s, df_s in sensor_dfs.items():
    print(f'\n=== {s} ===')
    print(df_s.round(2).to_string())